In [1]:
print("working fine")

working fine


In [2]:
%pwd

'f:\\PANTA\\Projects\\MedicalChatBot\\MedicalChatBot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'f:\\PANTA\\Projects\\MedicalChatBot\\MedicalChatBot'

In [5]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
# extract text from pdf files
def load_pdf_files(data_path):
    loader = DirectoryLoader(
        data_path,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents

In [7]:
extracted_data = load_pdf_files("data")
print(f"Number of pages loaded: {len(extracted_data)}")
extracted_data[1]

Number of pages loaded: 637


Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

In [8]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [9]:
minimal_docs = filter_to_minimal_docs(extracted_data)
minimal_docs[100]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='• T cell lymphocytes: 644-2200/mm\n3\n, 60-88% of all lym-\nphocytes.\n• B cell lymphocytes: 82-392/mm\n3\n, 3-20% of all lympho-\ncytes.\n• CD4+ lymphocytes: 500-1200/mm\n3\n, 34-67% of all\nlymphocytes.\nAbnormal results\nThe following results in AIDS tests indicate progres-\nsion of the disease:\n• Percentage of CD4+ lymphocytes: less than 20% of all\nlymphocytes.\n• CD4+ lymphocyte count: less than 200 cells/mm\n3\n.\n• Viral load test: Levels more than 5000 copies/mL.\n•/H9252-2-microglobulin: Levels more than 3.5 mg/dL.\n• P24 antigen: Measurable amounts in blood serum.\nResources\nBOOKS\nAvrameas, Stratis, and Therese Ternynck. “Enzyme Linked\nImmunosorbent Assay (ELISA).” In Encyclopedia of\nImmunology.V ol. 1. Ed. Ivan M. Roitt and Peter J. Delves.\nLondon: Academic Press, 1992.\nBennett, Rebecca, and Erin, Charles A. (Editors). HIV and\nAIDS Testing, Screening, and Confidentiality: Ethics, Law,\nand Social 

In [10]:
# split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)

    return texts_chunk

In [11]:
texts_chunk = text_split(minimal_docs=minimal_docs)
print(f"number of chunks: {len(texts_chunk)}")

number of chunks: 5860


In [12]:
texts_chunk[0]

Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

In [13]:
# from langchain.embeddings import HuggingFaceBgeEmbeddings

# def download_embeddings():
#     """
#     Download and return the HuggingFace embeddings model.
#     """
#     model_name = "sentence-transformers/all-MiniLM-L6-v2"
#     embeddings = HuggingFaceBgeEmbeddings(
#         model_name=model_name
#     )
#     return embeddings


# embedding model
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

def api_embeddings():

    embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    )   

    return embeddings


embedding = api_embeddings()

vector = embedding.embed_query(
    "Hello, How are you doing?"
)

print(vector[:5])

f:\PANTA\Projects\MedicalChatBot\MedicalChatBot\medbot_venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
f:\PANTA\Projects\MedicalChatBot\MedicalChatBot\medbot_venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[0.0029519866220653057, 0.03625120222568512, 0.0767618864774704, 0.05450519546866417, -0.029945313930511475]


In [14]:
vector = embedding.embed_query(
    "Hello, How are you doing?"
)

print(len(vector))
print(vector[:5])

384
[0.0029519866220653057, 0.03625120222568512, 0.0767618864774704, 0.05450519546866417, -0.029945313930511475]


In [15]:
# FAISS vector store
from langchain_community.vectorstores import FAISS

# create FAISS vector database from documents
docsearch = FAISS.from_documents(
    documents=texts_chunk,
    embedding=embedding
)

# save FAISS Index Locally
docsearch.save_local("faiss_index")


In [16]:
# save FAISS Index Locally
docsearch.save_local("faiss_index")

In [17]:
# load existing FAISS Index
docsearch = FAISS.load_local(
    "faiss_index",
    embeddings=embedding,
    allow_dangerous_deserialization=True
)


In [18]:
# Add more documents
docs = [
    Document(
        page_content="Hello, everybody! Hope you are doing well! If you have any feedback about this code drop me a mail. Thanks, Have a great learning!",
        metadata={"source": "YouTubeVideo"}
    )
]

docsearch.add_documents(docs)

#save again after adding documents
docsearch.save_local("faiss_index")

In [19]:
# retriever
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

retrieved_docs = retriever.invoke("What is Acne?")
print(retrieved_docs)

[Document(id='3a966723-e15c-426b-842d-e717221455ee', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='88dd894c-06db-485e-8705-4d6fadd5a086', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed.(Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'), Document(id='7b23ea35-8a00-4cfd-9c39-9ccb389c51b3', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with 

In [20]:
# Chat Model
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq

load_dotenv()

chatModel = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
    ) 

In [21]:
# RAG Chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)


In [22]:
# Correct Chain Creation
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

question_ans_chain = create_stuff_documents_chain(
    chatModel,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    question_ans_chain
)

In [23]:
# Query
response = rag_chain.invoke(
    {
        "input": "What is PCOS and Allergy?"
    }
)

print(response["answer"])

I don't know what PCOS is in relation to the provided context, but I can tell you that an allergy is a reaction of the immune system to a foreign substance, such as pollen or dust, which triggers the production of antibodies. Allergens are substances that provoke an allergic response, and they can include otherwise harmless substances that stimulate an immune reaction. I don't have information on PCOS in the provided context.


In [ ]:
# PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


# from pinecone import Pinecone
# pinecone_api_key = PINECONE_API_KEY

# pc = Pinecone(api_key=pinecone_api_key)


# from pinecone import ServerlessSpec

# index_name = "medical-chatbot"

# if not pc.has_index(index_name):
#     pc.create_index(
#         name = index_name,
#         dimension=384, # dimension of embeddings
#         metric="cosine" # cosine similarity
#         spec = ServerlessSpec(cloud="aws", region="us-east-1")
#     )

# index = pc.Index(index_name)


# from langchain_pinecone import PineconeVectorStore
# docsearch = PineconeVectorStore.from_documents(
#     documents=texts_chunk,
#     embedding=embedding,
#     index_name=index_name
# )


# # load existing index
# from langchain_pinecone import PineconeVectorStore
# # embed each chunk and upsert the embeddings into your Pinecone index
# docsearch = PineconeVectorStore.from_existing_index(
#     index_name=index_name,
#     embedding=embedding
# )


# # add more data to the existing Pinecone index
# docs = Document(
#     page_content="Hello, everybody! Hope you are doing well! If you have any feedback about this code drop me a mail. Thanks, Have a great learning!"
#     metadata={"source": "YouTubeVideo"}
# )

# docsearch.add_documents(documents=[docs])


# # retrive documents
# retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
# retrieved_docs = retriever.invoke("What is Acne?")

# retrieved_docs


# # refine the output - chatbot
# from langchain_openai import ChatOpenAI

# chatModel = ChatOpenAI(model="gpt-4o")

# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# from langchain_core.prompts import ChatPromptTemplate

# system_prompt = (
#     "You are an Medical assistant for question-answering tasks."
#     "Use the following pieces of retrieved context to answer the questions."
#     "If you don't know the answer, say that you don'y know."
#     "Use three sentences maximun and keep the answer concise."
#     "\n\n"
#     "{context}"
# )

# prompt = ChatPromptTemplate.from_messages(
#     [
#         ("system", system_prompt),
#         ("human", "{input}"),
#     ]
# )

# question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
# rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# response = rag_chain.invoke({"input": "What is PCOS and Allergy?"})
# print(response["answer"])